# LeetCode 1038 — Binary Search Tree to Greater Sum Tree bằng Python

Link đề: https://leetcode.com/problems/binary-search-tree-to-greater-sum-tree/ · Độ khó: **Medium** · Chủ đề: Tree (cây), Binary Search Tree (cây tìm kiếm nhị phân), DFS (duyệt theo chiều sâu), Binary Tree (cây nhị phân)

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def bstToGst(self, root: TreeNode) -> TreeNode:
        ...
```

Ghi chú: cho gốc của một **cây tìm kiếm nhị phân** (BST). Hãy biến nó thành **cây tổng lớn hơn** (Greater Sum Tree): mỗi nút có khóa mới bằng **khóa cũ cộng tổng mọi khóa lớn hơn** nó trong cây. Ý then chốt: duyệt thứ tự giữa (inorder) của BST cho các khóa **tăng dần**, nên nếu duyệt **ngược** (phải, gốc, trái) ta gặp các khóa **giảm dần**; giữ một **tổng chạy** các khóa đã gặp (đều lớn hơn nút hiện tại) rồi cộng khóa của nút hiện tại vào tổng để được giá trị mới. Bài này **tự dựng `TreeNode`** và hàm dựng/xuất cây kiểu LeetCode để kiểm thử; kết quả là một cây **duy nhất** nên so sánh đúng từng nút.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: khóa mới của nút = khóa cũ + tổng các khóa lớn hơn trong **toàn cây** (không chỉ trong cây con).
2. Nắm các công cụ Python cần thiết: lớp `TreeNode`, hàm dựng/xuất cây theo mức, đệ quy với biến `nonlocal`, ngăn xếp cho duyệt lặp, và duyệt **Morris** (dùng con trỏ rỗng để dựng "đường quay lại").
3. Hiểu nguyên lý **duyệt thứ tự giữa ngược** (reverse inorder): phải, gốc, trái cho các khóa giảm dần; tổng chạy `total` chính là tổng các khóa đã gặp.
4. Đi qua 4 cách: với mỗi nút cộng mọi khóa lớn hơn bằng cách quét toàn bộ (brute force), duyệt ngược bằng đệ quy với biến dùng chung (cách nộp chính), duyệt ngược bằng ngăn xếp (không đệ quy), và duyệt ngược Morris (`O(1)` bộ nhớ phụ).
5. Tránh các lỗi kinh điển: duyệt **xuôi** (trái, gốc, phải) cho tổng các khóa **nhỏ hơn**, quên cộng khóa **của chính nút**, truyền tổng **theo giá trị** (không chia sẻ giữa các lời gọi), cộng giá trị **mới** thay vì giá trị **cũ** vào tổng chạy, và nhầm thứ tự duyệt các con.

## 2. Đề bài & ví dụ

Cho `root` của một **cây tìm kiếm nhị phân** (BST). Hãy chuyển nó thành một **Cây Lớn Hơn** (Greater Tree) sao cho mỗi khóa của BST gốc được đổi thành **khóa gốc cộng với tổng mọi khóa lớn hơn khóa gốc** đó trong BST.

Nhắc lại: BST thỏa:

- cây con **trái** của một nút chỉ chứa các nút có khóa **nhỏ hơn** khóa của nút;
- cây con **phải** của một nút chỉ chứa các nút có khóa **lớn hơn** khóa của nút;
- cả hai cây con cũng là BST.

**Ví dụ 1:** `root = [4,1,6,0,2,5,7,null,null,null,3,null,null,null,8]` -> `[30,36,21,36,35,26,15,null,null,null,33,null,null,null,8]`.

**Ví dụ 2:** `root = [0,null,1]` -> `[1,null,1]`.

**Ràng buộc:**

- Số nút trong cây thuộc `[1, 100]`.
- `0 <= Node.val <= 100`.
- Mọi giá trị trong cây **khác nhau**.

Lưu ý: bài này giống bài 538 (Convert BST to Greater Tree).

In [1]:
from typing import List, Optional
from collections import deque
import copy
import random, time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(vals: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách theo mức kiểu LeetCode (None là nút trống; chỉ nút thật có hai phần tử con trong danh sách)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = deque([root]), 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

def tree_to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Xuất cây theo mức kiểu LeetCode (bỏ None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def inorder(root: Optional[TreeNode]) -> List[int]:
    out, stack, cur = [], [], root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        out.append(cur.val)
        cur = cur.right
    return out

def is_bst(root: Optional[TreeNode], lo=float("-inf"), hi=float("inf")) -> bool:
    if root is None:
        return True
    return lo < root.val < hi and is_bst(root.left, lo, root.val) and is_bst(root.right, root.val, hi)

def shape(root: Optional[TreeNode]):
    return None if root is None else (shape(root.left), shape(root.right))

examples = [
    ([4, 1, 6, 0, 2, 5, 7, None, None, None, 3, None, None, None, 8], [30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8]),
    ([0, None, 1], [1, None, 1]),
]
for vals, expected in examples:
    t = build_tree(vals)
    print(f"root={vals}\n   inorder = {inorder(t)}, là BST: {is_bst(t)}, kỳ vọng {expected}")
    assert tree_to_list(t) == vals and is_bst(t)

root=[4, 1, 6, 0, 2, 5, 7, None, None, None, 3, None, None, None, 8]
   inorder = [0, 1, 2, 3, 4, 5, 6, 7, 8], là BST: True, kỳ vọng [30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8]
root=[0, None, 1]
   inorder = [0, 1], là BST: True, kỳ vọng [1, None, 1]


## 3. Kiến thức nền

### 3.1. BST và duyệt thứ tự giữa

Duyệt **thứ tự giữa** (trái, gốc, phải) của một BST liệt kê các khóa **tăng dần**. Ngược lại duyệt **thứ tự giữa ngược** (phải, gốc, trái) liệt kê các khóa **giảm dần**. Đó là chìa khóa của bài.

In [2]:
t = build_tree(examples[0][0])
print("inorder (tăng dần):", inorder(t))
def reverse_inorder(node, out):
    if node:
        reverse_inorder(node.right, out)
        out.append(node.val)
        reverse_inorder(node.left, out)
    return out
print("reverse inorder (giảm dần):", reverse_inorder(t, []))

inorder (tăng dần): [0, 1, 2, 3, 4, 5, 6, 7, 8]
reverse inorder (giảm dần): [8, 7, 6, 5, 4, 3, 2, 1, 0]


### 3.2. Tổng chạy (running sum) khi duyệt ngược

Khi duyệt theo thứ tự giảm dần, tại thời điểm gặp một nút ta đã gặp **đúng tất cả** các khóa lớn hơn nó. Giữ biến `total` = tổng các khóa đã gặp; khi gặp nút mới: `total += node.val`, rồi gán `node.val = total` (gồm khóa cũ của chính nó và mọi khóa lớn hơn).

In [3]:
vals = reverse_inorder(t, [])
total, new = 0, []
for v in vals:
    total += v
    new.append(total)
print("các khóa giảm dần:", vals)
print("tổng chạy (giá trị mới) :", new)
print("sắp xếp theo khóa tăng dần:", sorted(zip(vals, new)))

các khóa giảm dần: [8, 7, 6, 5, 4, 3, 2, 1, 0]
tổng chạy (giá trị mới) : [8, 15, 21, 26, 30, 33, 35, 36, 36]
sắp xếp theo khóa tăng dần: [(0, 36), (1, 36), (2, 35), (3, 33), (4, 30), (5, 26), (6, 21), (7, 15), (8, 8)]


### 3.3. Biến dùng chung trong đệ quy

Muốn các lời gọi đệ quy **cùng cập nhật** một tổng, dùng biến `nonlocal` (hoặc thuộc tính đối tượng, hoặc danh sách một phần tử). Truyền tổng như **tham số** là truyền **theo giá trị**: thay đổi bên trong một lời gọi không được các lời gọi khác nhìn thấy.

In [4]:
def demo_nonlocal():
    total = 0
    def add(x):
        nonlocal total
        total += x
    for v in (3, 4, 5):
        add(v)
    return total

def demo_by_value():
    def add(total, x):
        total += x                  # chỉ đổi bản sao cục bộ
        return None
    total = 0
    for v in (3, 4, 5):
        add(total, v)
    return total

print("nonlocal:", demo_nonlocal(), "| tham số theo giá trị:", demo_by_value())

nonlocal: 12 | tham số theo giá trị: 0


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `[0, null, 1]` (gốc `0`, con phải `1`). Duyệt ngược: gặp `1` trước (`total = 1`, đổi thành `1`), rồi `0` (`total = 1 + 0 = 1`, đổi thành `1`). Kết quả `[1, null, 1]`.

Ví dụ 1: các khóa tăng dần `0, 1, 2, 3, 4, 5, 6, 7, 8`. Duyệt ngược `8, 7, ..., 0` với tổng chạy: `8, 15, 21, 26, 30, 33, 35, 36, 36`. Khóa `8 -> 8`, `7 -> 15`, `6 -> 21`, `5 -> 26`, `4 -> 30`, `3 -> 33`, `2 -> 35`, `1 -> 36`, `0 -> 36`. Cây kết quả giữ nguyên hình dạng, mỗi khóa thay bằng giá trị tương ứng: gốc `4 -> 30` ✓.

## 5. Cách 1 — Brute Force: mỗi nút cộng mọi khóa lớn hơn bằng cách quét

**Ý tưởng.** Thu thập danh sách mọi khóa của cây (không dùng tính chất BST). Với **mỗi nút** `node`, khóa mới là `sum(v for v in vals if v >= node.val)` (vì các giá trị khác nhau, `v >= node.val` gồm chính nó và mọi khóa lớn hơn). Đúng theo định nghĩa của đề nên là **oracle độc lập**; cũng dùng được cho cây không phải BST.

In [5]:
def collect(root: Optional[TreeNode], out: List[int]) -> List[int]:
    stack = [root]
    while stack:
        node = stack.pop()
        if node:
            out.append(node.val)
            stack.append(node.left)
            stack.append(node.right)
    return out

def gst_brute(root: Optional[TreeNode]) -> Optional[TreeNode]:
    vals = collect(root, [])
    stack = [root]
    while stack:
        node = stack.pop()
        if node:
            node.val = sum(v for v in vals if v >= node.val)       # vals là danh sách khóa GỐC (đã sao chép)
            stack.append(node.left)
            stack.append(node.right)
    return root

for vals, expected in examples:
    got = tree_to_list(gst_brute(build_tree(vals)))
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = [30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8] OK
brute = [1, None, 1] OK


**Độ phức tạp và nút thắt.** `O(n^2)` thời gian (với mỗi nút một lượt quét `n` khóa), `O(n)` bộ nhớ. Với `n <= 100` vẫn nhanh, nhưng ta **bỏ phí** tính chất BST: khóa lớn hơn của nút nằm ở **một phần xác định** của cây theo thứ tự duyệt. Đo thử khi `n` lớn hơn đề cho:

In [6]:
def random_bst(rng: random.Random, n: int, hi: int = 10 ** 6) -> Optional[TreeNode]:
    vals = rng.sample(range(0, hi + 1), n)
    root = None
    for v in vals:
        if root is None:
            root = TreeNode(v)
            continue
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None:
                    cur.left = TreeNode(v)
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(v)
                    break
                cur = cur.right
    return root

rng = random.Random(1038)
for n in (100, 400, 1600, 3200):
    t = random_bst(rng, n)
    t0 = time.perf_counter()
    gst_brute(t)
    dt = time.perf_counter() - t0
    print(f"n={n:5}: {dt * 1000:9.1f} ms")

n=  100:       0.6 ms
n=  400:       8.0 ms
n= 1600:     118.1 ms


n= 3200:     365.0 ms


Nút thắt: mỗi nút quét lại **toàn bộ** các khóa. Nhờ BST, các khóa lớn hơn là các khóa đứng **sau** nó theo thứ tự tăng dần: ta chỉ cần một lượt duyệt ngược với tổng chạy.

## 6. Ý tưởng thuật toán / bước đột phá

**Duyệt thứ tự giữa ngược (phải, gốc, trái) với tổng chạy dùng chung.**

```
reverse_inorder(node):
    nếu node rỗng: quay lại
    reverse_inorder(node.right)        # các khóa lớn hơn node trước
    total += node.val                  # cộng khóa của node
    node.val = total                   # khóa mới = khóa cũ + tổng các khóa lớn hơn
    reverse_inorder(node.left)         # rồi các khóa nhỏ hơn
```

Vì BST duyệt ngược cho khóa giảm dần, khi tới `node` thì `total` (trước khi cộng) đúng bằng tổng **mọi** khóa lớn hơn `node.val` trong toàn cây. Mỗi nút thăm một lần: `O(n)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy phải-gốc-trái với biến `nonlocal` (nộp) |
| 3 | Cùng thứ tự nhưng dùng ngăn xếp (không đệ quy) |
| 4 | Morris: dựng con trỏ tạm để quay lại, `O(1)` bộ nhớ phụ |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy phải-gốc-trái (cách nộp chính)

**Giải thích từng đoạn.**

- `total = 0`, khai báo `nonlocal total` trong hàm lồng để mọi lời gọi chia sẻ.
- Đệ quy sang con **phải** trước (các khóa lớn hơn).
- `total += node.val`; `node.val = total`.
- Rồi đệ quy sang con **trái**.
- Độ sâu đệ quy tối đa bằng chiều cao cây `<= 100`, an toàn.

In [7]:
def gst_recursive(root: Optional[TreeNode]) -> Optional[TreeNode]:
    total = 0

    def dfs(node: Optional[TreeNode]) -> None:
        nonlocal total
        if node is None:
            return
        dfs(node.right)               # các khóa lớn hơn
        total += node.val             # cộng khóa của chính nút
        node.val = total              # khóa mới
        dfs(node.left)                # các khóa nhỏ hơn

    dfs(root)
    return root

for vals, expected in examples:
    got = tree_to_list(gst_recursive(build_tree(vals)))
    print(f"đệ quy = {got}", "OK" if got == expected else "SAI")
    assert got == expected

đệ quy = [30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8] OK
đệ quy = [1, None, 1] OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def bstToGst(self, root: TreeNode) -> TreeNode:
        total = 0

        def dfs(node: TreeNode) -> None:
            nonlocal total
            if node is None:
                return
            dfs(node.right)                # duyệt các khóa lớn hơn trước
            total += node.val              # cộng khóa cũ của nút vào tổng chạy
            node.val = total              # khóa mới = khóa cũ + tổng các khóa lớn hơn
            dfs(node.left)                 # rồi các khóa nhỏ hơn

        dfs(root)
        return root

sol_recursive = Solution().bstToGst
for vals, expected in examples:
    assert tree_to_list(sol_recursive(build_tree(vals))) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n)`, bộ nhớ `O(h)` cho ngăn xếp đệ quy (`h` là chiều cao).

---

### 7.2. Cách 3 — Duyệt ngược bằng ngăn xếp

**Ý tưởng khác.** Mô phỏng đệ quy bằng ngăn xếp tường minh: đi xuống **phải** hết mức đẩy các nút vào ngăn xếp, lấy nút ra (xử lý), rồi chuyển sang con **trái** của nút vừa lấy.

**Giải thích từng đoạn.**

- `cur` là nút đang xét; `while cur or stack`.
- `while cur: stack.append(cur); cur = cur.right`: đi sâu bên phải.
- `cur = stack.pop()`: xử lý nút (`total += cur.val; cur.val = total`).
- `cur = cur.left`: sang con trái.

In [9]:
def gst_stack(root: Optional[TreeNode]) -> Optional[TreeNode]:
    total = 0
    stack: List[TreeNode] = []
    cur = root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.right
        cur = stack.pop()
        total += cur.val
        cur.val = total
        cur = cur.left
    return root

for vals, expected in examples:
    got = tree_to_list(gst_stack(build_tree(vals)))
    print(f"ngăn xếp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

ngăn xếp = [30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8] OK
ngăn xếp = [1, None, 1] OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def bstToGst(self, root: TreeNode) -> TreeNode:
        total = 0
        stack = []
        cur = root
        while cur or stack:
            while cur:                    # đi sâu sang phải (các khóa lớn hơn)
                stack.append(cur)
                cur = cur.right
            cur = stack.pop()
            total += cur.val              # cộng khóa của nút
            cur.val = total               # khóa mới
            cur = cur.left                # rồi sang cây con trái
        return root

sol_stack = Solution().bstToGst
for vals, expected in examples:
    assert tree_to_list(sol_stack(build_tree(vals))) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n)`, bộ nhớ `O(h)` cho ngăn xếp.

---

### 7.3. Cách 4 — Duyệt ngược Morris (`O(1)` bộ nhớ phụ)

**Ý tưởng khác.** Duyệt Morris dựng **con trỏ tạm** từ nút "đứng ngay trước" (theo thứ tự duyệt) về nút hiện tại để quay lại mà không dùng ngăn xếp. Với duyệt **ngược**, nút đứng ngay **trước** `cur` trong thứ tự (phải, gốc, trái) là nút **trái nhất** của cây con **phải** của `cur` (hậu duệ nhỏ nhất bên phải). Ta nối `succ.left = cur`, đi xuống phải; khi quay lại qua con trỏ tạm thì gỡ nó đi và xử lý `cur`.

**Giải thích từng đoạn.**

- Nếu `cur.right` rỗng: xử lý `cur` (`total += cur.val; cur.val = total`) rồi sang `cur.left`.
- Ngược lại tìm `succ` = nút trái nhất của `cur.right` (dừng nếu `succ.left` rỗng hoặc đã trỏ về `cur`).
  - `succ.left is None` (lần đầu): `succ.left = cur`, đi xuống `cur = cur.right`.
  - `succ.left is cur` (quay lại): gỡ `succ.left = None`, xử lý `cur`, sang `cur = cur.left`.
- Sau khi xong, mọi con trỏ tạm đã được gỡ nên cấu trúc cây nguyên vẹn.

In [11]:
def gst_morris(root: Optional[TreeNode]) -> Optional[TreeNode]:
    total = 0
    cur = root
    while cur:
        if cur.right is None:
            total += cur.val
            cur.val = total
            cur = cur.left
        else:
            succ = cur.right
            while succ.left and succ.left is not cur:
                succ = succ.left                      # nút trái nhất của cây con phải
            if succ.left is None:
                succ.left = cur                       # dựng con trỏ tạm để quay lại
                cur = cur.right
            else:
                succ.left = None                      # quay lại: gỡ con trỏ tạm
                total += cur.val
                cur.val = total
                cur = cur.left
    return root

for vals, expected in examples:
    got = tree_to_list(gst_morris(build_tree(vals)))
    print(f"Morris = {got}", "OK" if got == expected else "SAI")
    assert got == expected

Morris = [30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8] OK
Morris = [1, None, 1] OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def bstToGst(self, root: TreeNode) -> TreeNode:
        total = 0
        cur = root
        while cur:
            if cur.right is None:             # không còn khóa lớn hơn chưa xử lý bên phải
                total += cur.val
                cur.val = total
                cur = cur.left
            else:
                succ = cur.right
                while succ.left and succ.left is not cur:
                    succ = succ.left          # nút trái nhất của cây con phải: nút xử lý ngay trước cur
                if succ.left is None:
                    succ.left = cur           # nối con trỏ tạm về cur
                    cur = cur.right
                else:
                    succ.left = None          # đã quay lại: gỡ con trỏ tạm rồi xử lý cur
                    total += cur.val
                    cur.val = total
                    cur = cur.left
        return root

sol_morris = Solution().bstToGst
for vals, expected in examples:
    assert tree_to_list(sol_morris(build_tree(vals))) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n)` (mỗi cạnh đi tối đa hai lần), bộ nhớ phụ `O(1)`.

## 8. TRACE từng bước

### 8.1. Đệ quy cho ví dụ 2 `[0, null, 1]`

In từng lần xử lý nút với `total` trước và sau.

In [13]:
def trace_recursive(root):
    total = 0
    def dfs(node, depth):
        nonlocal total
        pad = "  " * depth
        if node is None:
            return
        print(f"{pad}vào nút {node.val}: sang con phải")
        dfs(node.right, depth + 1)
        before = total
        total += node.val
        print(f"{pad}xử lý nút {node.val}: total {before} + {node.val} = {total} -> khóa mới {total}")
        old = node.val
        node.val = total
        dfs(node.left, depth + 1)
    dfs(root, 0)
    return root

assert tree_to_list(trace_recursive(build_tree([0, None, 1]))) == [1, None, 1]

vào nút 0: sang con phải
  vào nút 1: sang con phải
  xử lý nút 1: total 0 + 1 = 1 -> khóa mới 1
xử lý nút 0: total 1 + 0 = 1 -> khóa mới 1


### 8.2. Morris cho cây `[2, 1, 3]`

In từng bước: con trỏ tạm được dựng và gỡ ở đâu.

In [14]:
def trace_morris(root):
    total = 0
    cur = root
    while cur:
        if cur.right is None:
            total += cur.val
            print(f"cur={cur.val}: không có con phải -> xử lý, total = {total}")
            cur.val = total
            cur = cur.left
        else:
            succ = cur.right
            while succ.left and succ.left is not cur:
                succ = succ.left
            if succ.left is None:
                print(f"cur={cur.val}: dựng con trỏ tạm từ nút {succ.val} về {cur.val}, đi sang phải")
                succ.left = cur
                cur = cur.right
            else:
                print(f"cur={cur.val}: quay lại qua con trỏ tạm (gỡ), xử lý")
                succ.left = None
                total += cur.val
                cur.val = total
                print(f"   total = {total}")
                cur = cur.left
    return root

t = trace_morris(build_tree([2, 1, 3]))
print("kết quả:", tree_to_list(t))
assert tree_to_list(t) == [5, 6, 3]

cur=2: dựng con trỏ tạm từ nút 3 về 2, đi sang phải
cur=3: không có con phải -> xử lý, total = 3
cur=2: quay lại qua con trỏ tạm (gỡ), xử lý
   total = 5
cur=1: không có con phải -> xử lý, total = 6
kết quả: [5, 6, 3]


### 8.3. Duyệt xuôi cho tổng sai

Nếu duyệt xuôi (trái, gốc, phải) với tổng chạy, mỗi nút nhận tổng các khóa **nhỏ hơn hoặc bằng** nó (tổng tiền tố), không phải tổng các khóa lớn hơn.

In [15]:
def forward_wrong(root):
    total = 0
    def dfs(node):
        nonlocal total
        if node:
            dfs(node.left)
            total += node.val
            node.val = total
            dfs(node.right)
    dfs(root)
    return root

print("đúng :", tree_to_list(Solution().bstToGst(build_tree([2, 1, 3]))))
print("duyệt xuôi (sai):", tree_to_list(forward_wrong(build_tree([2, 1, 3]))))
assert tree_to_list(forward_wrong(build_tree([2, 1, 3]))) != tree_to_list(Solution().bstToGst(build_tree([2, 1, 3])))

đúng : [5, 6, 3]
duyệt xuôi (sai): [3, 1, 6]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (đệ quy phải-gốc-trái): ngắn gọn, `O(n)`, độ sâu tối đa `100`.

```python
class Solution:
    def bstToGst(self, root: TreeNode) -> TreeNode:
        total = 0

        def dfs(node: TreeNode) -> None:
            nonlocal total
            if node is None:
                return
            dfs(node.right)                # duyệt các khóa lớn hơn trước
            total += node.val              # cộng khóa cũ của nút vào tổng chạy
            node.val = total              # khóa mới = khóa cũ + tổng các khóa lớn hơn
            dfs(node.left)                 # rồi các khóa nhỏ hơn

        dfs(root)
        return root
```

In [16]:
class Solution:
    def bstToGst(self, root: TreeNode) -> TreeNode:
        total = 0

        def dfs(node: TreeNode) -> None:
            nonlocal total
            if node is None:
                return
            dfs(node.right)                # duyệt các khóa lớn hơn trước
            total += node.val              # cộng khóa cũ của nút vào tổng chạy
            node.val = total              # khóa mới = khóa cũ + tổng các khóa lớn hơn
            dfs(node.left)                 # rồi các khóa nhỏ hơn

        dfs(root)
        return root

sol = Solution()
for vals, expected in examples:
    got = tree_to_list(sol.bstToGst(build_tree(vals)))
    print(f"{got}", "OK" if got == expected else "SAI")
    assert got == expected

[30, 36, 21, 36, 35, 26, 15, None, None, None, 33, None, None, None, 8] OK
[1, None, 1] OK


## 10. Vì sao đúng

**Thứ tự duyệt.** Với BST, mọi khóa trong cây con trái `<` khóa gốc `<` mọi khóa trong cây con phải. Duyệt **phải, gốc, trái** thăm cây con phải (mọi khóa lớn hơn) rồi gốc rồi cây con trái (mọi khóa nhỏ hơn); áp dụng quy nạp cho từng cây con cho thấy toàn bộ cây được thăm theo thứ tự khóa **giảm dần** (đây là duyệt thứ tự giữa viết ngược).

**Bất biến.** Ngay trước khi xử lý nút `u`, `total` bằng tổng các khóa **gốc** của mọi nút đã được xử lý, tức mọi nút có khóa lớn hơn `u.val` (vì thứ tự giảm dần, và mỗi nút chỉ được cộng **khóa gốc** của nó vì ta cộng `total += node.val` trước khi gán khóa mới). Sau khi xử lý, `u.val = total + u.val_gốc = khóa gốc + tổng các khóa lớn hơn`: đúng định nghĩa. `total` mới bằng tổng các khóa gốc `>= u.val_gốc`.

**Dùng khóa gốc.** Phép gán `node.val = total` xảy ra **sau** khi đã cộng `node.val` (khóa gốc) vào `total`, nên `total` chỉ chứa các khóa gốc (không chứa giá trị mới của các nút trước).

**Cách 3** thực hiện cùng thứ tự bằng ngăn xếp. **Cách 4 (Morris)** thực hiện cùng thứ tự: con trỏ tạm `succ.left = cur` với `succ` là nút trái nhất của cây con phải của `cur` (nút được thăm ngay trước `cur` theo thứ tự phải-gốc-trái), cho phép quay lại `cur` sau khi xong cây con phải; lần quay lại thứ hai ta gỡ con trỏ để khôi phục cây.

**Độ phức tạp.** Mỗi nút được xử lý một lần: `O(n)`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) tính trực tiếp `sum(v for v in vals if v >= node.val)` cho từng nút, không dùng tính chất BST. Ta đối chiếu bốn cách trên rất nhiều BST ngẫu nhiên có seed (n tới `100`, kể cả cây lệch hoàn toàn), kiểm tra **hình dạng cây không đổi** và các con trỏ tạm của Morris đã được gỡ hết.

In [17]:
rng = random.Random(1010)
for _ in range(400):
    n = rng.randint(1, 100)
    root = random_bst(rng, n, 100)
    lst = tree_to_list(root)
    ref = tree_to_list(gst_brute(build_tree(lst)))
    for name, fn in (("recursive", gst_recursive), ("stack", gst_stack), ("morris", gst_morris),
                     ("sol_recursive", sol_recursive), ("sol_stack", sol_stack), ("sol_morris", sol_morris), ("final", sol.bstToGst)):
        t = fn(build_tree(lst))
        assert tree_to_list(t) == ref, (name, lst)
        assert shape(t) == shape(build_tree(lst))
print("400 BST ngẫu nhiên (n tới 100): 7 cách khớp oracle brute và giữ nguyên hình dạng cây")

400 BST ngẫu nhiên (n tới 100): 7 cách khớp oracle brute và giữ nguyên hình dạng cây


## 11. Time / Space Complexity

| Cách | Time | Space (phụ) |
|---|---|---|
| 1. Quét mọi khóa cho từng nút | `O(n^2)` | `O(n)` |
| 2. Đệ quy phải-gốc-trái | `O(n)` | `O(h)` |
| 3. Ngăn xếp | `O(n)` | `O(h)` |
| 4. Morris | `O(n)` | `O(1)` |

Đo ở cỡ tối đa của đề (`n = 100`) và ở cỡ lớn hơn (`n = 10^4`) để thấy quan hệ tuyến tính; với cây lệch (chuỗi) dùng cách lặp/Morris (đệ quy sâu `n`):

In [18]:
rng = random.Random(11)
def chain_right(n):
    root = TreeNode(0)
    cur = root
    for v in range(1, n):
        cur.right = TreeNode(v)
        cur = cur.right
    return root

def chain_left(n):
    root = TreeNode(n - 1)
    cur = root
    for v in range(n - 2, -1, -1):
        cur.left = TreeNode(v)
        cur = cur.left
    return root

cases = {"ngẫu nhiên n=100": lambda: random_bst(rng, 100, 100), "chuỗi phải n=100": lambda: chain_right(100), "chuỗi trái n=100": lambda: chain_left(100),
         "ngẫu nhiên n=10^4": lambda: random_bst(rng, 10 ** 4, 10 ** 6)}
for label, make in cases.items():
    lst = tree_to_list(make())
    ref = tree_to_list(gst_stack(build_tree(lst)))
    row = [f"{label:20}"]
    for name, fn in (("đệ quy", sol_recursive), ("ngăn xếp", sol_stack), ("Morris", sol_morris)):
        if name == "đệ quy" and "10^4" in label:
            continue
        t = build_tree(lst)
        t0 = time.perf_counter()
        got = fn(t)
        dt = (time.perf_counter() - t0) * 1000
        assert tree_to_list(got) == ref, (label, name)
        row.append(f"{name} {dt:7.2f} ms")
    print(" | ".join(row))

ngẫu nhiên n=100     | đệ quy    0.01 ms | ngăn xếp    0.02 ms | Morris    0.02 ms
chuỗi phải n=100     | đệ quy    0.02 ms | ngăn xếp    0.01 ms | Morris    0.02 ms
chuỗi trái n=100     | đệ quy    0.01 ms | ngăn xếp    0.01 ms | Morris    0.01 ms
ngẫu nhiên n=10^4    | ngăn xếp    1.63 ms | Morris    2.58 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc) và các tính chất. Kết quả là cây **duy nhất** nên so sánh **chính xác** bằng `tree_to_list`.

In [19]:
def all_solutions():
    return {"recursive": gst_recursive, "stack": gst_stack, "morris": gst_morris,
            "sol_recursive": sol_recursive, "sol_stack": sol_stack, "sol_morris": sol_morris, "final": sol.bstToGst}

def check_all(lst, expected=None):
    ref = tree_to_list(gst_brute(build_tree(lst)))
    for name, fn in all_solutions().items():
        t = fn(build_tree(lst))
        assert tree_to_list(t) == ref, (name, lst, tree_to_list(t), ref)
        assert shape(t) == shape(build_tree(lst))
    if expected is not None:
        assert ref == expected, (lst, ref, expected)
    return ref

# 12.1 ví dụ đề
for vals, expected in examples:
    check_all(vals, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([5], [5]),                                      # một nút: chính nó
    ([0], [0]),                                      # khóa 0
    ([2, 1], [2, 3]),                                # gốc 2, con trái 1: 2 -> 2, 1 -> 1 + 2 = 3
    ([1, None, 2], [3, None, 2]),                    # gốc 1, con phải 2: 2 -> 2, 1 -> 1 + 2 = 3
    ([2, 1, 3], [5, 6, 3]),                          # tính tay ở bài tập 2: 3 -> 3, 2 -> 5, 1 -> 6
    ([0, None, 1], [1, None, 1]),                    # ví dụ 2
    ([3, 2, None, 1], [3, 5, None, 6]),              # chuỗi trái 3 > 2 > 1: 3 -> 3, 2 -> 5, 1 -> 6
    ([1, None, 2, None, 3], [6, None, 5, None, 3]),  # chuỗi phải 1 < 2 < 3: 3 -> 3, 2 -> 5, 1 -> 6
    ([100], [100]),                                  # khóa biên
    ([50, 25, 75, 10, 30, 60, 90], [275, 330, 165, 340, 305, 225, 90]),   # cây đầy đủ 7 nút: khóa tăng dần 10..90 có tổng hậu tố 340, 330, 305, 275, 225, 165, 90
]
for lst, expected in edge:
    check_all(lst, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 10 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm trên các BST nhỏ ngẫu nhiên, so với oracle brute)
def wrong_forward(root):                            # duyệt xuôi (trái, gốc, phải): tổng các khóa nhỏ hơn
    return forward_wrong(root)

def wrong_exclude_self(root):                       # khóa mới = tổng các khóa lớn hơn (không cộng khóa của chính nút)
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return
        dfs(node.right)
        old = node.val
        node.val = total                            # chưa cộng khóa của chính nút
        total += old
        dfs(node.left)
    dfs(root)
    return root

def wrong_by_value(root):                           # truyền tổng theo giá trị: các lời gọi không chia sẻ
    def dfs(node, total):
        if node is None:
            return total
        total = dfs(node.right, total)
        total += node.val
        node.val = total
        dfs(node.left, 0)                           # sai: quên truyền tổng tiếp tục cho con trái
        return total
    dfs(root, 0)
    return root

def wrong_add_new_value(root):                      # cộng GIÁ TRỊ MỚI vào tổng chạy (cộng dồn lặp)
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return
        dfs(node.right)
        node.val += total
        total += node.val                           # sai: cộng giá trị mới (đã gồm total) vào total
        dfs(node.left)
    dfs(root)
    return root

def wrong_left_first(root):                         # phải-trái-gốc: thứ tự duyệt sai
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return
        dfs(node.right)
        dfs(node.left)
        total += node.val
        node.val = total
    dfs(root)
    return root

def find_counter(wrong, tries=20000, seed=9):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 7)
        lst = tree_to_list(random_bst(rng, n, 9))
        if tree_to_list(wrong(build_tree(lst))) != tree_to_list(gst_brute(build_tree(lst))):
            return lst
    return None

counters = {}
for name, fn in (("duyệt xuôi (trái, gốc, phải)", wrong_forward), ("quên cộng khóa của chính nút", wrong_exclude_self),
                 ("truyền tổng theo giá trị", wrong_by_value), ("cộng giá trị mới vào tổng chạy", wrong_add_new_value),
                 ("thứ tự phải-trái-gốc", wrong_left_first)):
    lst = find_counter(fn)
    assert lst is not None, name
    counters[name] = lst
    print(f"{name:34} root={lst}: sai -> {tree_to_list(fn(build_tree(lst)))}, đúng -> {tree_to_list(gst_brute(build_tree(lst)))}")
    check_all(lst)
print("12.3 phản ví dụ: OK")

duyệt xuôi (trái, gốc, phải)       root=[9, 5, None, 4, 6, 1]: sai -> [25, 10, None, 5, 16, 1], đúng -> [9, 20, None, 24, 15, 25]
quên cộng khóa của chính nút       root=[9, 5, None, 4, 6, 1]: sai -> [0, 15, None, 20, 9, 24], đúng -> [9, 20, None, 24, 15, 25]
truyền tổng theo giá trị           root=[9, 5, None, 4, 6, 1]: sai -> [9, 11, None, 4, 6, 1], đúng -> [9, 20, None, 24, 15, 25]
cộng giá trị mới vào tổng chạy     root=[9, 5, None, 4, 6, 1]: sai -> [9, 29, None, 57, 15, 111], đúng -> [9, 20, None, 24, 15, 25]
thứ tự phải-trái-gốc               root=[9, 5, None, 4, 6, 1]: sai -> [25, 16, None, 11, 6, 7], đúng -> [9, 20, None, 24, 15, 25]
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên: BST với n tới 100, giá trị trong [0, 100]; và cây lệch hoàn toàn
rng = random.Random(4038)
for _ in range(500):
    n = rng.randint(1, 100)
    check_all(tree_to_list(random_bst(rng, n, 100)))
for n in (1, 2, 50, 100):
    check_all(tree_to_list(chain_right(n)))
    check_all(tree_to_list(chain_left(n)))
print("12.4 ngẫu nhiên và cây lệch: OK")

# 12.5 tính chất: giữ nguyên hình dạng; khóa mới của nút nhỏ nhất = tổng mọi khóa; khóa mới của nút lớn nhất = chính nó;
# mọi khóa mới đơn điệu theo khóa gốc (nghịch); inorder mới giảm dần; hiệu hai khóa mới liên tiếp (theo khóa gốc) = khóa gốc của nút nhỏ hơn
rng = random.Random(5038)
for _ in range(300):
    n = rng.randint(1, 100)
    root = random_bst(rng, n, 100)
    orig = inorder(root)
    t = sol.bstToGst(root)
    new = inorder(t)
    assert new[0] == sum(orig) and new[-1] == orig[-1]
    assert all(new[i] - new[i + 1] == orig[i] for i in range(len(new) - 1))
    assert new == [sum(orig[i:]) for i in range(len(orig))]
print("12.5 tính chất (suffix sum theo inorder): OK")

# 12.6 Morris khôi phục cấu trúc: sau khi chạy không còn vòng lặp (duyệt hữu hạn) và số nút giữ nguyên
rng = random.Random(6038)
for _ in range(200):
    n = rng.randint(1, 100)
    lst = tree_to_list(random_bst(rng, n, 100))
    t = sol_morris(build_tree(lst))
    count, stack = 0, [t]
    while stack:
        node = stack.pop()
        if node:
            count += 1
            assert count <= n
            stack.append(node.left); stack.append(node.right)
    assert count == n
print("12.6 Morris không để lại con trỏ tạm: OK")

12.4 ngẫu nhiên và cây lệch: OK
12.5 tính chất (suffix sum theo inorder): OK
12.6 Morris không để lại con trỏ tạm: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Duyệt xuôi (trái, gốc, phải)

Duyệt xuôi cho tổng các khóa **nhỏ hơn hoặc bằng** (tổng tiền tố), không phải các khóa lớn hơn.

In [22]:
lst = counters["duyệt xuôi (trái, gốc, phải)"]
print("root =", lst, "| sai ->", tree_to_list(wrong_forward(build_tree(lst))), "| đúng ->", tree_to_list(Solution().bstToGst(build_tree(lst))))
assert tree_to_list(wrong_forward(build_tree(lst))) != tree_to_list(Solution().bstToGst(build_tree(lst)))

root = [9, 5, None, 4, 6, 1] | sai -> [25, 10, None, 5, 16, 1] | đúng -> [9, 20, None, 24, 15, 25]


Vì sao sai: các khóa lớn hơn nằm **phía phải** trong thứ tự giữa. **Đúng:** duyệt phải, gốc, trái.

### Lỗi 2 — Quên cộng khóa của chính nút

Đề yêu cầu **khóa gốc cộng với** tổng các khóa lớn hơn; bỏ khóa gốc làm thiếu đúng giá trị đó.

In [23]:
lst = counters["quên cộng khóa của chính nút"]
print("root =", lst, "| sai ->", tree_to_list(wrong_exclude_self(build_tree(lst))), "| đúng ->", tree_to_list(Solution().bstToGst(build_tree(lst))))
assert tree_to_list(wrong_exclude_self(build_tree(lst))) != tree_to_list(Solution().bstToGst(build_tree(lst)))

root = [9, 5, None, 4, 6, 1] | sai -> [0, 15, None, 20, 9, 24] | đúng -> [9, 20, None, 24, 15, 25]


Vì sao sai: ví dụ 2 cho `[1, null, 1]` (nút `1` giữ `1`), còn cách sai cho nút lớn nhất `0`. **Đúng:** `total += node.val; node.val = total`.

### Lỗi 3 — Truyền tổng theo giá trị

Một tham số số nguyên được sao chép khi gọi hàm: các lời gọi con không cập nhật tổng của cha. Cần biến dùng chung (`nonlocal`, thuộc tính, hoặc trả về tổng).

In [24]:
lst = counters["truyền tổng theo giá trị"]
print("root =", lst, "| sai ->", tree_to_list(wrong_by_value(build_tree(lst))), "| đúng ->", tree_to_list(Solution().bstToGst(build_tree(lst))))
assert tree_to_list(wrong_by_value(build_tree(lst))) != tree_to_list(Solution().bstToGst(build_tree(lst)))

root = [9, 5, None, 4, 6, 1] | sai -> [9, 11, None, 4, 6, 1] | đúng -> [9, 20, None, 24, 15, 25]


Vì sao sai: cây con trái cần biết tổng của cây con phải và của chính nút. **Đúng:** `nonlocal total`.

### Lỗi 4 — Cộng giá trị **mới** vào tổng chạy

Sau `node.val += total`, `node.val` đã chứa `total`; cộng nó vào `total` đếm các khóa lớn hơn nhiều lần.

In [25]:
lst = counters["cộng giá trị mới vào tổng chạy"]
print("root =", lst, "| sai ->", tree_to_list(wrong_add_new_value(build_tree(lst))), "| đúng ->", tree_to_list(Solution().bstToGst(build_tree(lst))))
assert tree_to_list(wrong_add_new_value(build_tree(lst))) != tree_to_list(Solution().bstToGst(build_tree(lst)))

root = [9, 5, None, 4, 6, 1] | sai -> [9, 29, None, 57, 15, 111] | đúng -> [9, 20, None, 24, 15, 25]


Vì sao sai: `total` phải là tổng các khóa **gốc**. **Đúng:** `total += node.val` (khóa gốc) **trước** rồi `node.val = total`.

### Lỗi 5 — Nhầm thứ tự duyệt các con

Phải xử lý nút **giữa** hai lần duyệt: sau cây con phải và **trước** cây con trái. Xử lý nút sau cả hai con (hậu tự) cộng sai.

In [26]:
lst = counters["thứ tự phải-trái-gốc"]
print("root =", lst, "| sai ->", tree_to_list(wrong_left_first(build_tree(lst))), "| đúng ->", tree_to_list(Solution().bstToGst(build_tree(lst))))
assert tree_to_list(wrong_left_first(build_tree(lst))) != tree_to_list(Solution().bstToGst(build_tree(lst)))

root = [9, 5, None, 4, 6, 1] | sai -> [25, 16, None, 11, 6, 7] | đúng -> [9, 20, None, 24, 15, 25]


Vì sao sai: khóa của cây con trái phải được cộng **sau** nút hiện tại. **Đúng:** `phải -> gốc -> trái`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "mỗi khóa cộng với tổng các khóa lớn hơn/nhỏ hơn" trên BST ⇒ **duyệt thứ tự giữa** (xuôi hoặc ngược) với **tổng chạy**.
- Muốn các khóa **lớn hơn**: duyệt ngược `phải -> gốc -> trái`. Muốn các khóa **nhỏ hơn**: duyệt xuôi `trái -> gốc -> phải`.
- Biến tổng phải được **chia sẻ** giữa các lời gọi (`nonlocal`, thuộc tính, hoặc trả về).
- Cộng **khóa gốc** vào tổng (không phải khóa mới).
- Không muốn đệ quy: ngăn xếp, hoặc **Morris** nếu cần `O(1)` bộ nhớ phụ.
- Cùng họ: Convert BST to Greater Tree (538), Kth Smallest Element in a BST (230), Validate BST (98), Range Sum of BST (938).
- Test nên có: một nút, chuỗi trái, chuỗi phải, cây cân bằng, khóa `0`, `n = 100`.

## 15. Biến thể

1. **Cây nhị phân bất kỳ (không phải BST):** dùng sắp xếp các khóa rồi tổng hậu tố (bài tập 5), hoặc quét như Cách 1.
2. **Cây tổng nhỏ hơn** (smaller sum tree): duyệt xuôi, khóa mới = khóa gốc + tổng các khóa nhỏ hơn.
3. **Có khóa trùng nhau:** quy ước rõ "lớn hơn" (nghiêm ngặt) để xử lý nhóm khóa bằng nhau.
4. Bài liên quan: 538, 230, 938.

Biến thể 2 (cây tổng nhỏ hơn) kiểm bằng oracle quét:

In [27]:
def smaller_sum_tree(root: Optional[TreeNode]) -> Optional[TreeNode]:
    total = 0
    def dfs(node):
        nonlocal total
        if node is None:
            return
        dfs(node.left)
        total += node.val
        node.val = total
        dfs(node.right)
    dfs(root)
    return root

def smaller_sum_brute(root):
    vals = collect(root, [])
    stack = [root]
    while stack:
        node = stack.pop()
        if node:
            node.val = sum(v for v in vals if v <= node.val)
            stack.append(node.left); stack.append(node.right)
    return root

rng = random.Random(15)
for _ in range(200):
    lst = tree_to_list(random_bst(rng, rng.randint(1, 30), 100))
    assert tree_to_list(smaller_sum_tree(build_tree(lst))) == tree_to_list(smaller_sum_brute(build_tree(lst)))
print("biến thể cây tổng nhỏ hơn: duyệt xuôi khớp oracle quét")

biến thể cây tổng nhỏ hơn: duyệt xuôi khớp oracle quét


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét mọi khóa | Theo định nghĩa | `O(n^2)` | `O(n)` | Oracle, dùng được cho cây bất kỳ |
| 2. Đệ quy phải-gốc-trái | Tổng chạy `nonlocal` | `O(n)` | `O(h)` | **Cách nộp chính** |
| 3. Ngăn xếp | Cùng thứ tự, không đệ quy | `O(n)` | `O(h)` | Cây rất sâu |
| 4. Morris | Con trỏ tạm | `O(n)` | `O(1)` | Cần bộ nhớ phụ `O(1)` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
t = build_tree([2, 1, 3])
print(inorder(t), reverse_inorder(t, []))
print(tree_to_list(Solution().bstToGst(build_tree([1, None, 2]))))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # hai danh sách
my_guess_2 = None      # danh sách theo mức

### Bài 2 — Tính tay (dễ-vừa)

`root = [2, 1, 3]`. Ghi danh sách theo mức của cây sau khi chuyển thành cây tổng lớn hơn.

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành đệ quy.

In [30]:
def exercise_3(root: Optional[TreeNode]) -> Optional[TreeNode]:
    total = 0

    def dfs(node: Optional[TreeNode]) -> None:
        nonlocal total
        # TODO 1: node rỗng thì quay lại
        # TODO 2: duyệt cây con PHẢI trước
        # TODO 3: total += node.val; node.val = total
        # TODO 4: duyệt cây con TRÁI
        pass

    dfs(root)
    return root


# for vals, expected in examples:
#     assert tree_to_list(exercise_3(build_tree(vals))) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(root)` bằng **ngăn xếp** (duyệt phải, gốc, trái, không đệ quy).

In [31]:
def exercise_4(root: Optional[TreeNode]) -> Optional[TreeNode]:
    # TODO
    pass


# for vals, expected in examples:
#     assert tree_to_list(exercise_4(build_tree(vals))) == expected

### Bài 5 — Mini challenge: cây nhị phân bất kỳ (khó)

Viết `exercise_5(root)` làm cùng phép biến đổi (khóa mới = khóa cũ + tổng các khóa **lớn hơn**) nhưng cho **cây nhị phân bất kỳ** (không nhất thiết là BST); các khóa khác nhau.

```python
tree_to_list(exercise_5(build_tree([3, 1, 2])))        -> [3, 6, 5]
tree_to_list(exercise_5(build_tree([1, 2, 3])))        -> [6, 5, 3]
tree_to_list(exercise_5(build_tree([4, 1, 6, 0, 2, 5, 7, None, None, None, 3, None, None, None, 8])))   # như ví dụ 1
```
Gợi ý: không dùng được thứ tự duyệt của BST; sắp xếp các khóa rồi dùng tổng hậu tố, hoặc quét.

In [32]:
def exercise_5(root: Optional[TreeNode]) -> Optional[TreeNode]:
    # TODO
    pass


# assert tree_to_list(exercise_5(build_tree([3, 1, 2]))) == [3, 6, 5]
# assert tree_to_list(exercise_5(build_tree([1, 2, 3]))) == [6, 5, 3]
# assert tree_to_list(exercise_5(build_tree(examples[0][0]))) == examples[0][1]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `inorder` tăng dần; `reverse_inorder` giảm dần.
- Mức 2: cây `[2, 1, 3]`: inorder `[1, 2, 3]`; reverse `[3, 2, 1]`.
- Mức 3: `[1, 2, 3] [3, 2, 1]`; `[1, null, 2]` -> `[3, null, 2]`.

**Bài 2**
- Mức 1: duyệt ngược `3, 2, 1` với tổng chạy.
- Mức 2: `3 -> 3`, `2 -> 3 + 2 = 5`, `1 -> 5 + 1 = 6`.
- Mức 3: giữ hình dạng: gốc `2 -> 5`, con trái `1 -> 6`, con phải `3 -> 3`: `[5, 6, 3]`.

**Bài 3**
- Mức 1: `if node is None: return`.
- Mức 2: `dfs(node.right)`.
- Mức 3: `total += node.val; node.val = total; dfs(node.left)`.

**Bài 4**
- Mức 1: `while cur or stack:` đi sâu sang phải bằng `stack.append(cur); cur = cur.right`.
- Mức 2: `cur = stack.pop()`; `total += cur.val`; `cur.val = total`.
- Mức 3: `cur = cur.left`.

**Bài 5**
- Mức 1: thu thập các khóa, sắp xếp giảm dần.
- Mức 2: tổng hậu tố: `new[v] = tổng các khóa >= v`.
- Mức 3: duyệt cây và gán `node.val = new[node.val]`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
t = build_tree([2, 1, 3])
print(inorder(t), reverse_inorder(t, []))
print(tree_to_list(Solution().bstToGst(build_tree([1, None, 2]))))

assert (inorder(t), reverse_inorder(t, [])) == ([1, 2, 3], [3, 2, 1])
assert tree_to_list(Solution().bstToGst(build_tree([1, None, 2]))) == [3, None, 2]

[1, 2, 3] [3, 2, 1]
[3, None, 2]


### Đáp án Bài 2

Duyệt ngược `3, 2, 1`: `3 -> 3`, `2 -> 5`, `1 -> 6`. Hình dạng giữ nguyên: `[5, 6, 3]`.

In [34]:
hand_answer_solution = [5, 6, 3]
assert tree_to_list(Solution().bstToGst(build_tree([2, 1, 3]))) == hand_answer_solution
assert tree_to_list(gst_brute(build_tree([2, 1, 3]))) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(root: Optional[TreeNode]) -> Optional[TreeNode]:
    total = 0

    def dfs(node: Optional[TreeNode]) -> None:
        nonlocal total
        if node is None:
            return
        dfs(node.right)
        total += node.val
        node.val = total
        dfs(node.left)

    dfs(root)
    return root

for vals, expected in examples:
    assert tree_to_list(exercise_3_solution(build_tree(vals))) == expected
rng = random.Random(3)
for _ in range(200):
    lst = tree_to_list(random_bst(rng, rng.randint(1, 30), 100))
    assert tree_to_list(exercise_3_solution(build_tree(lst))) == tree_to_list(gst_brute(build_tree(lst)))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(root: Optional[TreeNode]) -> Optional[TreeNode]:
    total = 0
    stack = []
    cur = root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.right
        cur = stack.pop()
        total += cur.val
        cur.val = total
        cur = cur.left
    return root

for vals, expected in examples:
    assert tree_to_list(exercise_4_solution(build_tree(vals))) == expected
rng = random.Random(4)
for _ in range(200):
    lst = tree_to_list(random_bst(rng, rng.randint(1, 30), 100))
    assert tree_to_list(exercise_4_solution(build_tree(lst))) == tree_to_list(gst_brute(build_tree(lst)))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(root: Optional[TreeNode]) -> Optional[TreeNode]:
    vals = sorted(collect(root, []), reverse=True)
    suffix = {}
    total = 0
    for v in vals:
        total += v
        suffix[v] = total
    stack = [root]
    while stack:
        node = stack.pop()
        if node:
            node.val = suffix[node.val]
            stack.append(node.left)
            stack.append(node.right)
    return root

def random_binary_tree(rng, n, hi=100):
    vals = rng.sample(range(0, hi + 1), n)
    root = TreeNode(vals[0])
    nodes = [root]
    slots = [(root, "left"), (root, "right")]
    for v in vals[1:]:
        parent, side = slots.pop(rng.randrange(len(slots)))
        child = TreeNode(v)
        setattr(parent, side, child)
        slots += [(child, "left"), (child, "right")]
    return root

assert tree_to_list(exercise_5_solution(build_tree([3, 1, 2]))) == [3, 6, 5]
assert tree_to_list(exercise_5_solution(build_tree([1, 2, 3]))) == [6, 5, 3]
assert tree_to_list(exercise_5_solution(build_tree(examples[0][0]))) == examples[0][1]
rng = random.Random(5)
for _ in range(300):
    lst = tree_to_list(random_binary_tree(rng, rng.randint(1, 30)))
    assert tree_to_list(exercise_5_solution(build_tree(lst))) == tree_to_list(gst_brute(build_tree(lst)))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (duyệt phải-gốc-trái với tổng chạy)

```python
class Solution:
    def bstToGst(self, root: TreeNode) -> TreeNode:
        total = 0

        def dfs(node: TreeNode) -> None:
            nonlocal total
            if node is None:
                return
            dfs(node.right)                # các khóa lớn hơn trước
            total += node.val              # cộng khóa cũ
            node.val = total              # khóa mới = khóa cũ + tổng các khóa lớn hơn
            dfs(node.left)                 # rồi các khóa nhỏ hơn

        dfs(root)
        return root
```

Công thức nhớ nhanh: BST duyệt **phải, gốc, trái** cho khóa **giảm dần**; `total += node.val; node.val = total`.

### Checklist trước khi submit

- [ ] Duyệt theo thứ tự **phải → gốc → trái** (không xuôi)?
- [ ] Cộng **khóa gốc** của nút vào tổng rồi mới gán khóa mới (không bỏ khóa của chính nút, không cộng khóa mới)?
- [ ] Tổng chạy được **chia sẻ** (`nonlocal`) giữa các lời gọi?
- [ ] Trả về `root` (cây được sửa tại chỗ)?
- [ ] Nếu cần tránh đệ quy: ngăn xếp hoặc Morris (gỡ hết con trỏ tạm)?
- [ ] Đã thử: một nút, chuỗi trái, chuỗi phải, khóa `0`, ví dụ 1-2, `n = 100`?